In [ ]:
import os
import shutil
import ee
import numpy as np
import pandas as pd
from google.colab import files

try:
    OUTPUT_DIR = 'SoilMate_Export'
    MASTER_DIR = os.path.join(OUTPUT_DIR, 'Master Data')
    SAMPLE_DIR = os.path.join(OUTPUT_DIR, 'Sample Data')
    os.makedirs(MASTER_DIR, exist_ok=True)
    os.makedirs(SAMPLE_DIR, exist_ok=True)

    MY_PROJECT_ID = 'proven-cosmos-479318-j9'
    try:
        ee.Initialize(project=MY_PROJECT_ID)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=MY_PROJECT_ID)

    print("✅ Cell 1 succeeded: folders created, GEE connection established.")
except Exception as e:
    print(f"❌ Cell 1 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    LATITUDE, LONGITUDE = 37.0000, 35.3213    # Cukurova region
    START_DATE = '2000-01-01'
    END_DATE   = '2026-01-01'                  # Exclusive, must be a year start

    roi = ee.Geometry.Point([LONGITUDE, LATITUDE]).buffer(5000)   # 5 km buffer

    def extract_time_series(collection, bands, scale, chunk_years=4):
        """Extract a time series from an ImageCollection.
        Splits the range into chunks to stay under GEE's 5000-element limit."""
        start_year = int(START_DATE[:4])
        end_year   = int(END_DATE[:4])

        def reduce_region(image):
            stats = image.select(bands).reduceRegion(
                reducer=ee.Reducer.mean(), geometry=roi, scale=scale, maxPixels=1e13)
            return ee.Feature(None, stats).set('date', image.date().format('YYYY-MM-dd'))

        data_list = []
        for year in range(start_year, end_year, chunk_years):
            chunk_start = f"{year}-01-01"
            chunk_end   = f"{min(year + chunk_years, end_year)}-01-01"
            chunk_col   = collection.filterDate(chunk_start, chunk_end)

            try:
                size = chunk_col.size().getInfo()
            except Exception:
                size = 0
            if size == 0:
                continue

            print(f"  -> {chunk_start} to {chunk_end} ({size} images)")
            features = chunk_col.map(reduce_region).getInfo()['features']
            data_list.extend(f['properties'] for f in features)

        df = pd.DataFrame(data_list)
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df = df.drop_duplicates(subset=['date']).set_index('date').sort_index()
        return df

    print("✅ Cell 2 succeeded: parameters and helper function defined.")
except Exception as e:
    print(f"❌ Cell 2 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching ERA5-Land...")
    era5_bands = [
        'temperature_2m', 'dewpoint_temperature_2m', 'total_precipitation_sum',
        'surface_solar_radiation_downwards_sum', 'total_evaporation_sum', 'potential_evaporation_sum',
        'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2',
        'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4'
    ]
    era5_col = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").filterBounds(roi)
    df_era5 = extract_time_series(era5_col, era5_bands, scale=11000, chunk_years=5)

    if df_era5.empty:
        raise ValueError("ERA5 returned no data.")

    # Unit conversions
    df_era5['temperature_2m']          -= 273.15
    df_era5['dewpoint_temperature_2m'] -= 273.15
    for col in ['total_precipitation_sum', 'total_evaporation_sum', 'potential_evaporation_sum']:
        df_era5[col] *= 1000                                    # m -> mm
    df_era5['surface_solar_radiation_downwards_sum'] /= 86400   # J/m2 -> W/m2

    print(f"✅ Cell 3 succeeded: ERA5 has {len(df_era5)} rows and {len(df_era5.columns)} columns.")
except Exception as e:
    print(f"❌ Cell 3 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching MODIS NDVI/EVI...")

    def mask_vi(img):
        # SummaryQA: 0 = good, 1 = marginal, 2 = snow/ice, 3 = cloudy
        return img.updateMask(img.select('SummaryQA').lte(1))

    modis_vi_col = ee.ImageCollection("MODIS/061/MOD13Q1").filterBounds(roi).map(mask_vi)
    df_modis_vi = extract_time_series(modis_vi_col, ['NDVI', 'EVI'], scale=250, chunk_years=26)

    if df_modis_vi.empty:
        raise ValueError("MODIS NDVI/EVI returned no data.")

    df_modis_vi[['NDVI', 'EVI']] *= 0.0001   # scale factor

    print(f"✅ Cell 4 succeeded: NDVI/EVI has {len(df_modis_vi)} rows.")
except Exception as e:
    print(f"❌ Cell 4 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching MODIS LST...")
    modis_lst_col = ee.ImageCollection("MODIS/061/MOD11A2").filterBounds(roi)
    df_lst = extract_time_series(modis_lst_col, ['LST_Day_1km', 'LST_Night_1km'], scale=1000, chunk_years=26)

    if df_lst.empty:
        raise ValueError("MODIS LST returned no data.")

    df_lst['LST_Day_1km']   = df_lst['LST_Day_1km']   * 0.02 - 273.15   # Kelvin -> Celsius
    df_lst['LST_Night_1km'] = df_lst['LST_Night_1km'] * 0.02 - 273.15

    print(f"✅ Cell 5 succeeded: LST has {len(df_lst)} rows.")
except Exception as e:
    print(f"❌ Cell 5 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Computing MODIS NDWI...")

    def add_ndwi_modis(img):
        clear = img.select('StateQA').bitwiseAnd(3).eq(0)                                  # cloud-free pixels
        ndwi  = img.normalizedDifference(['sur_refl_b02', 'sur_refl_b06']).rename('NDWI')  # NIR vs SWIR
        return ndwi.updateMask(clear).copyProperties(img, ['system:time_start'])

    ndwi_col = ee.ImageCollection("MODIS/061/MOD09A1").filterBounds(roi).map(add_ndwi_modis)
    df_ndwi = extract_time_series(ndwi_col, ['NDWI'], scale=500, chunk_years=26)

    if df_ndwi.empty:
        raise ValueError("NDWI returned no data.")

    print(f"✅ Cell 6 succeeded: NDWI has {len(df_ndwi)} rows.")
except Exception as e:
    print(f"❌ Cell 6 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching static data...")
    elevation = ee.Image("USGS/SRTMGL1_003").select('elevation')
    slope = ee.Terrain.slope(elevation).rename('slope')
    clay  = ee.Image("projects/soilgrids-isric/clay_mean").select(0).rename('clay')
    sand  = ee.Image("projects/soilgrids-isric/sand_mean").select(0).rename('sand')
    phh2o = ee.Image("projects/soilgrids-isric/phh2o_mean").select(0).rename('phh2o')

    static_stats = ee.Image([elevation, slope, clay, sand, phh2o]).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=roi, scale=250, maxPixels=1e13).getInfo()
    static_stats['phh2o'] = static_stats['phh2o'] / 10     # pH scale correction

    print(static_stats)
    print(f"✅ Cell 7 succeeded: {len(static_stats)} static variables fetched.")
except Exception as e:
    print(f"❌ Cell 7 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    dfs = [df_era5, df_modis_vi, df_lst, df_ndwi]
    common_start = max(d.index.min() for d in dfs)     # first day all sources have data

    df_final = df_era5.astype(float).join([df_modis_vi, df_lst, df_ndwi], how='outer')
    df_final = df_final.interpolate(method='time')     # time-based, handles irregular intervals
    df_final = df_final.loc[common_start:].ffill()     # no fake back-fill, only the tail end is filled

    for key, value in static_stats.items():
        df_final[key] = float(value) if value is not None else np.nan

    df_final = df_final.round(4)
    df_final.index.name = 'Date'

    remaining_nans = df_final.isna().sum()
    remaining_nans = remaining_nans[remaining_nans > 0]
    if len(remaining_nans) > 0:
        print("Remaining NaN counts:\n", remaining_nans)

    master_path = os.path.join(MASTER_DIR, f"SoilMate_MasterData_{START_DATE}_to_{END_DATE}.csv")
    df_final.to_csv(master_path)

    print(f"Rows: {len(df_final)}, Columns: {len(df_final.columns)}, "
          f"Range: {df_final.index.min().date()} -> {df_final.index.max().date()}")
    print(f"✅ Cell 8 succeeded: master data saved to {master_path}")
except Exception as e:
    print(f"❌ Cell 8 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    df_daily = pd.read_csv(master_path, parse_dates=['Date'], index_col='Date')

    grouping_days = [3, 5, 7, 10, 14, 15, 20, 30]

    agg_dict = {col: 'mean' for col in df_daily.columns}
    for col in ['total_precipitation_sum', 'total_evaporation_sum', 'potential_evaporation_sum']:
        agg_dict[col] = 'sum'          # only cumulative variables are summed

    for days in grouping_days:
        rule  = 'MS' if days == 30 else f'{days}D'      # 30 = calendar month
        label = "Monthly (calendar month)" if days == 30 else f"{days}-day"

        df_grouped = df_daily.resample(rule).agg(agg_dict).round(4).dropna()
        df_grouped.to_csv(os.path.join(SAMPLE_DIR, f"grouped_{days}_days.csv"))
        print(f"  {label}: {len(df_grouped)} rows")

    print(f"✅ Cell 9 succeeded: {len(grouping_days)} grouped datasets saved.")
except Exception as e:
    print(f"❌ Cell 9 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    zip_name = 'SoilMate_Export'
    shutil.make_archive(zip_name, 'zip', root_dir='.', base_dir=OUTPUT_DIR)
    files.download(f"{zip_name}.zip")

    print(f"✅ Cell 10 succeeded: {zip_name}.zip created, download started.")
except Exception as e:
    print(f"❌ Cell 10 failed: {type(e).__name__}: {e}")